In [0]:
CATALOG = "uc13"
SCHEMA = "ingestion"
COMPANY_NAME = dbutils.secrets.get("uc13", "sp_company_name")
VOLUME_PATH = f"/Volumes/uc13/ingestion/raw_files/{COMPANY_NAME}"
TABLE_RELEVANCE = f"{CATALOG}.classification.doc_relevance"

# Crear schema si no existe
spark.sql("CREATE SCHEMA IF NOT EXISTS uc13.classification")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_RELEVANCE} (
  file_name     STRING,
  relative_path STRING,
  folder        STRING,
  workstream    STRING,
  should_parse  BOOLEAN,
  reason        STRING,
  confidence    STRING,
  classified_at TIMESTAMP
) USING DELTA
""")

print(f"Company: {COMPANY_NAME}")
print(f"Volume: {VOLUME_PATH}")
print("Setup done")

In [0]:
## Listing all files in volume:
import os
from collections import Counter

all_files = []
for root, dirs, files in os.walk(VOLUME_PATH):
    for f in files:
        if not f.startswith("."):
            full_path = os.path.join(root, f)
            relative = full_path.replace(VOLUME_PATH + "/", "")
            folder = relative.split("/")[0] if "/" in relative else "root"
            all_files.append({
                "file_name": f,
                "relative_path": relative,
                "folder": folder
            })

print(f"Total files: {len(all_files)}")
folders = Counter(f["folder"] for f in all_files)
for folder, count in sorted(folders.items()):
    print(f"  {folder}: {count}")

In [0]:
import json
import re
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

def classify_batch(files_batch):

    def sanitize(name):
        return (name
            .replace('"', "'")
            .replace('\\', '/')
            .replace('\n', ' ')
            .replace('\u2013', '-')
            .replace('\u2014', '-')
            .replace('–', '-')
            .replace('—', '-')
            .replace('\u00a0', ' ')
            .replace('(', '[')
            .replace(')', ']')
        )

    file_list = "\n".join([
        f"{i+1}. [folder: {f['folder']}] {sanitize(f['file_name'])}"
        for i, f in enumerate(files_batch)
    ])

    prompt = """Classify each file from a PE due diligence data room.

IMPORTANT: Use ONLY these exact values for each field.

workstream: financial, legal, operations, hr, insurance, it, ma, noise

priority_tier: 1, 2, 3, or null
- Tier 1 (highest value, parse first — at least one per workstream):
  financial: CIM, QofE, Financial Model, Projection Model, audited P&L,
             Balance Sheet, EBITDA bridge, Addbacks, Forecast model,
             Internal KPI dashboard, FDD databook, company Tax returns,
             Revenue by customer, CAP Table, Capitalization table
  ma: CIM, diligence workbooks, FDD databooks, investor presentations
  legal: primary service agreements, main customer contracts, key NDAs,
         corporate governance docs, legal entity structure
  operations: primary SOP, company org chart, main operational KPI report
  hr: primary employee handbook, main company policy document
  insurance: primary insurance policy summary
  it: IT systems overview, main software license summary

- Tier 2 (high value):
  financial: pipeline report, backlog, secondary financial schedules
  ma: data trackers, index reports, deal materials
  legal: secondary contracts, vendor agreements, compliance docs
  operations: secondary SOPs, operational reports, process docs
  hr: secondary handbooks, role descriptions, compensation policies
  insurance: individual certificates, secondary policies
  it: IT asset lists, secondary software licenses

- Tier 3 (useful but not critical):
  Bank statements, reconciliations, payroll summaries, checking accounts,
  individual caregiver contracts (when many identical ones exist),
  secondary employee policies, operational checklists

- null: ONLY when should_parse=false.
  RULE: if should_parse=true, priority_tier MUST be 1, 2, or 3. NEVER null when should_parse=true.

should_parse=false — EXCLUSIONARY CRITERIA (do not parse these):
  - Weekly or bi-weekly payroll detail PDFs
  - Individual employee records, I-9, W-4, hiring packets, onboarding forms
  - Blank form templates
  - ZIP files, images (.jpeg, .png, .gif)
  - Binary Excel files (.xlsb)
  - Monthly individual bank statements (when there are dozens of them)
  - Individual caregiver or staff contracts (when there are many identical ones)
  - Personal employee files, medical records, background checks

should_parse=true: all Tier 1, Tier 2, and Tier 3 documents

confidence must be the string "high", "medium", or "low" — never a number.
workstream values must be lowercase exactly as listed above.

Return ONLY a JSON array, no markdown, no explanation:
[{"workstream":"ma","should_parse":true,"priority_tier":1,"confidence":"high","reason":"CIM document"}]

Files:
""" + file_list

    response = client.predict(
        endpoint="databricks-meta-llama-3-3-70b-instruct",
        inputs={
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 3000,
            "temperature": 0.0
        }
    )

    text = response["choices"][0]["message"]["content"].strip()
    text = re.sub(r'```json\s*|\s*```', '', text).strip()

    try:
        result = json.loads(text)
        if isinstance(result, list) and len(result) > 0:
            valid_ws = ["financial","legal","operations","hr","insurance","it","ma","noise"]
            for r in result:
                if isinstance(r.get("confidence"), (int, float)):
                    r["confidence"] = "high" if r["confidence"] >= 0.7 else "medium" if r["confidence"] >= 0.4 else "low"
                ws = r.get("workstream", "noise").lower()
                r["workstream"] = ws if ws in valid_ws else "noise"
                tier = r.get("priority_tier")
                r["priority_tier"] = int(tier) if tier in [1, 2, 3] else None

            while len(result) < len(files_batch):
                result.append({
                    "workstream": "needs_review",
                    "should_parse": False,
                    "priority_tier": None,
                    "confidence": "low",
                    "reason": "missing from LLM response"
                })

            return result[:len(files_batch)]
    except Exception as e:
        print(f"  Parse error: {e} | Response: {text[:300]}")
    return None

# Test con 10 archivos
test_result = classify_batch(all_files[:10])
if test_result:
    for f, r in zip(all_files[:10], test_result):
        print(f"{f['file_name'][:50]:<50} | {r['workstream']:<12} | T{r['priority_tier']} | parse:{str(r['should_parse']):<5} | {r['confidence']}")
else:
    print("ERROR")

In [0]:
BATCH_SIZE = 20
all_results = []
failed_batches = []
total_batches = -(-len(all_files) // BATCH_SIZE)

for batch_start in range(0, len(all_files), BATCH_SIZE):
    batch = all_files[batch_start:batch_start + BATCH_SIZE]
    batch_num = batch_start // BATCH_SIZE + 1

    result = classify_batch(batch)

    if result and len(result) == len(batch):
        for f, r in zip(batch, result):
            all_results.append({
                **f,
                "workstream": r.get("workstream", "noise"),
                "should_parse": r.get("should_parse", False),
                "priority_tier": r.get("priority_tier"),
                "confidence": r.get("confidence", "low"),
                "reason": r.get("reason", ""),
            })
        print(f"Batch {batch_num}/{total_batches}: {len(batch)} files ✓")
    else:
        failed_batches.append(batch_start)
        print(f"Batch {batch_num}/{total_batches}: retrying individually...")
        for single_file in batch:
            single_result = classify_batch([single_file])
            if single_result and len(single_result) == 1:
                r = single_result[0]
                all_results.append({
                    **single_file,
                    "workstream": r.get("workstream", "noise"),
                    "should_parse": r.get("should_parse", False),
                    "priority_tier": r.get("priority_tier"),
                    "confidence": r.get("confidence", "low"),
                    "reason": r.get("reason", ""),
                })
            else:
                all_results.append({
                    **single_file,
                    "workstream": "needs_review",
                    "should_parse": False,
                    "priority_tier": None,
                    "confidence": "low",
                    "reason": "classification failed - requires manual review",
                })
        print(f"Batch {batch_num}/{total_batches}: individual retry done")

print(f"\nTotal classified: {len(all_results)}")
print(f"Batches that needed retry: {len(failed_batches)}")

In [0]:
import re
from collections import defaultdict

def get_base_name(file_name):
    stem = os.path.splitext(file_name)[0]
    clean = re.sub(
        r'[-_\s]*(v\w+|\d{4}[-_]\w+[-_]\d+|\d{1,2}[._]\d{1,2}[._]\d{2,4}|'
        r'vSHARE_[\d.]+|vRevised|vF|vUPLOAD|final|updated|copy|'
        r'\(\d+\)|Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[-_\s]*',
        '', stem, flags=re.IGNORECASE
    ).strip()
    return clean.lower()

groups = defaultdict(list)
for r in all_results:
    base = get_base_name(r["file_name"])
    key = f"{r['folder']}|{base}"
    groups[key].append(r)

final_results = []
dedupe_count = 0

for key, group in groups.items():
    if len(group) == 1:
        final_results.append(group[0])
    else:
        final_results.append(group[0])
        for older in group[1:]:
            final_results.append({
                **older,
                "should_parse": False,
                "reason": f"older version of: {group[0]['file_name']}",
                "confidence": "high"
            })
            dedupe_count += 1

print(f"Total files: {len(final_results)}")
print(f"Marked as older versions: {dedupe_count}")
print(f"Will parse: {sum(1 for r in final_results if r['should_parse'])}")
print(f"Will skip:  {sum(1 for r in final_results if not r['should_parse'])}")
print(f"Tier 1: {sum(1 for r in final_results if r['priority_tier'] == 1)}")
print(f"Tier 2: {sum(1 for r in final_results if r['priority_tier'] == 2)}")
print(f"Tier 3: {sum(1 for r in final_results if r['priority_tier'] == 3)}")

from collections import Counter
ws_dist = Counter(r["workstream"] for r in final_results if r["should_parse"])
print(f"\nTo parse by workstream:")
for ws, count in sorted(ws_dist.items(), key=lambda x: -x[1]):
    print(f"  {ws}: {count}")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, IntegerType, TimestampType
)
from datetime import datetime

schema = StructType([
    StructField("file_name",     StringType(),  False),
    StructField("relative_path", StringType(),  False),
    StructField("folder",        StringType(),  True),
    StructField("workstream",    StringType(),  True),
    StructField("should_parse",  BooleanType(), False),
    StructField("priority_tier", IntegerType(), True),
    StructField("reason",        StringType(),  True),
    StructField("confidence",    StringType(),  True),
    StructField("classified_at", TimestampType(), False),
])

rows = [
    Row(
        file_name=r["file_name"],
        relative_path=r["relative_path"],
        folder=r["folder"],
        workstream=r["workstream"],
        should_parse=bool(r["should_parse"]),
        priority_tier=r.get("priority_tier"),
        reason=r["reason"],
        confidence=r["confidence"],
        classified_at=datetime.now()
    )
    for r in final_results
]

df = spark.createDataFrame(rows, schema=schema)
df.write.mode("overwrite").saveAsTable(TABLE_RELEVANCE)
print(f"Saved {df.count()} classifications to {TABLE_RELEVANCE}")

In [0]:
%sql
SELECT 
  priority_tier,
  workstream,
  COUNT(*) as total,
  SUM(CASE WHEN should_parse THEN 1 ELSE 0 END) as to_parse
FROM uc13.classification.doc_relevance
GROUP BY priority_tier, workstream
ORDER BY priority_tier, workstream

In [0]:
%sql
SELECT file_name, folder, reason
FROM uc13.classification.doc_relevance
WHERE priority_tier IS NULL AND should_parse = true

In [0]:
%sql
UPDATE uc13.classification.doc_relevance
SET should_parse = false
WHERE file_name = 'Reconciliation_0422_0121.pdf';
SELECT 'Fixed' as status;

In [0]:
%sql
SELECT file_name, folder, workstream, should_parse, reason, confidence, priority_tier, classified_at FROM uc13.classification.doc_relevance LIMIT 20

In [0]:
# %sql
# TRUNCATE TABLE uc13.classification.doc_relevance;
# SELECT 'Cleared' as status;